# ICM (Intrinsic Curiosity Module) - FFAI Assistant

Forward + Inverse + Feature models para recompensa intrínseca.

In [ ]:
import tensorflow as tf
from tensorflow import keras
print(f"TensorFlow: {tf.__version__}")

In [ ]:
STATE_SIZE, ACTION_SIZE, FEATURE_DIM = 256, 15, 64

# Feature Extractor: State -> Features
feature_net = keras.Sequential([
    keras.layers.Dense(128, activation='relu', input_shape=(STATE_SIZE,)),
    keras.layers.Dense(FEATURE_DIM, activation='relu')
], name='ICM_Feature')

# Forward Model: (Feature, Action) -> Next Feature
forward_input = keras.Input(shape=(FEATURE_DIM + ACTION_SIZE,))
x = keras.layers.Dense(128, activation='relu')(forward_input)
forward_output = keras.layers.Dense(FEATURE_DIM)(x)
forward_net = keras.Model(forward_input, forward_output, name='ICM_Forward')

# Inverse Model: (Feature, Next Feature) -> Action
inverse_input = keras.Input(shape=(FEATURE_DIM * 2,))
x = keras.layers.Dense(128, activation='relu')(inverse_input)
inverse_output = keras.layers.Dense(ACTION_SIZE)(x)
inverse_net = keras.Model(inverse_input, inverse_output, name='ICM_Inverse')

print(f"✓ Feature: {feature_net.count_params():,} params")
print(f"✓ Forward: {forward_net.count_params():,} params")
print(f"✓ Inverse: {inverse_net.count_params():,} params")

In [ ]:
# Exportar
models = {
    'icm_feature': feature_net,
    'icm_forward': forward_net,
    'icm_inverse': inverse_net
}

for name, model in models.items():
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    with open(f'{name}.tflite', 'wb') as f:
        f.write(converter.convert())
    print(f"✓ Exportado: {name}.tflite")

In [ ]:
from google.colab import files
for name in models.keys():
    files.download(f'{name}.tflite')